In [104]:
#Extract packages
import pandas as pd
import numpy as np

In [105]:
#Extract dataset
df = pd.read_csv('https://raw.githubusercontent.com/HumayDS/A15---24-Reqemsal-data-analitika/refs/heads/main/user_activity_growth_2.csv')

In [106]:
#date
#new_users
#sessions – number of sessions
#acquisition_channel - the channel through which the customer was acquired
#avg_session_duration – average session duration (minutes)
df.head()

,date,acquisition_channel,new_users,active_users,sessions,avg_session_duration
0,2025-01-01,organic,37.0,75,140.0,8.9
1,2025-01-01,paid_ads,14.0,28,53.0,7.5
2,2025-01-01,social_media,18.0,36,56.0,6.1
3,2025-01-01,referral,30.0,70,88.0,7.5
4,2025-01-01,email,12.0,33,53.0,3.8


In [107]:
#Category analysis
df['acquisition_channel'].unique()

array(['organic', 'paid_ads', 'social_media', 'referral', 'email',
       'socialmedia'], dtype=object)

In [108]:
#Unify the categories
df['acquisition_channel'] = df['acquisition_channel'].replace({
    'socialmedia': 'social_media'
})

In [109]:
# Count of columns and rows
df.shape

(2298, 6)

In [110]:
#Check na's
df.isnull().sum()

,0
date,0
acquisition_channel,0
new_users,22
active_users,0
sessions,22
avg_session_duration,0


In [111]:
#Handle with filling median
df = df.fillna(df.median(numeric_only=True))

In [112]:
#Check again
df.isnull().sum()

,0
date,0
acquisition_channel,0
new_users,0
active_users,0
sessions,0
avg_session_duration,0


In [113]:
#Check dublicates
#Duplicate rows in a dataset are records (rows) that contain exactly the same values across all columns as another row.
df.duplicated().sum()

np.int64(22)

In [114]:
# Overview dublicates
df.groupby(['date', 'acquisition_channel']).size().reset_index(name='count').query('count > 1')

,date,acquisition_channel,count
7,2025-01-02,paid_ads,2
41,2025-01-09,organic,2
57,2025-01-12,paid_ads,2
78,2025-01-16,social_media,2
97,2025-01-20,social_media,2
107,2025-01-22,social_media,2
181,2025-02-06,social_media,2
300,2025-03-02,social_media,2
514,2025-04-14,social_media,2
540,2025-04-20,email,2


In [115]:
#Delete dublicates
df = df.drop_duplicates()

In [116]:
#Check types of columns
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2276 entries, 0 to 2275
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   date                  2276 non-null   object 
 1   acquisition_channel   2276 non-null   object 
 2   new_users             2276 non-null   float64
 3   active_users          2276 non-null   int64  
 4   sessions              2276 non-null   float64
 5   avg_session_duration  2276 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 124.5+ KB


In [117]:
#Convert to time format
df['date'] = pd.to_datetime(df['date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2276 entries, 0 to 2275
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  2276 non-null   datetime64[ns]
 1   acquisition_channel   2276 non-null   object        
 2   new_users             2276 non-null   float64       
 3   active_users          2276 non-null   int64         
 4   sessions              2276 non-null   float64       
 5   avg_session_duration  2276 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 124.5+ KB


In [118]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,2276,2025-08-16 00:20:14.762741760,2025-01-01 00:00:00,2025-04-24 00:00:00,2025-08-16 00:00:00,2025-12-08 00:00:00,2026-03-31 00:00:00,NaN
new_users,2276.0,64.501318,-264.0,20.75,47.0,87.25,500.0,63.877668
active_users,2276.0,179.566344,-26.0,54.0,128.0,240.0,1414.0,178.190577
sessions,2276.0,282.848858,-42.0,85.0,196.0,370.25,4030.0,300.370738
avg_session_duration,2276.0,6.846485,3.5,5.2,6.9,8.5,10.0,1.881571


In [119]:
#Let’s look at the start and end periods
start_date = df['date'].min()
end_date = df['date'].max()

print(f'Start Date: {start_date}')
print(f'End Date: {end_date}')

Start Date: 2025-01-01 00:00:00
End Date: 2026-03-31 00:00:00


In [120]:
#Create month and year columns
df['month'] =  df['date'].dt.month
df['year'] = df['date'].dt.year

In [121]:
#Outlier check
num_cols = ['new_users', 'active_users', 'sessions', 'avg_session_duration']
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outlier_mask = (df[num_cols] < (Q1 - 2 * IQR)) | (df[num_cols] > (Q3 + 2 * IQR))

In [122]:
#Outlier check
outlier_mask.sum()

,0
new_users,80
active_users,84
sessions,79
avg_session_duration,0


In [123]:
df_outliers = df[outlier_mask.any(axis=1)]
df_outliers.head(20)

,date,acquisition_channel,new_users,active_users,sessions,avg_session_duration,month,year
63,2025-01-13,referral,42.0,135,2210.0,8.8,1,2025
428,2025-03-27,referral,25.0,53,950.0,7.0,3,2025
690,2025-05-19,organic,56.0,182,2460.0,5.1,5,2025
1140,2025-08-17,organic,35.0,113,1680.0,4.0,8,2025
1148,2025-08-18,referral,29.0,65,1090.0,7.7,8,2025
1195,2025-08-28,organic,211.0,551,1007.0,9.3,8,2025
1250,2025-09-08,organic,172.0,527,991.0,8.6,9,2025
1361,2025-09-30,paid_ads,190.0,651,1064.0,8.1,9,2025
1423,2025-10-12,referral,216.0,746,914.0,5.6,10,2025
1430,2025-10-14,organic,176.0,588,1110.0,7.6,10,2025


#Rule
##Outliers are of two types: Real events (should not be removed) and data errors (should be removed).
##Before removing an outlier, you must be able to explain its cause.
##If you cannot explain it:

##either keep it
##or flag it separately

In [124]:
df['is_outlier'] = outlier_mask.any(axis=1)

In [125]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,2276,2025-08-16 00:20:14.762741760,2025-01-01 00:00:00,2025-04-24 00:00:00,2025-08-16 00:00:00,2025-12-08 00:00:00,2026-03-31 00:00:00,NaN
new_users,2276.0,64.501318,-264.0,20.75,47.0,87.25,500.0,63.877668
active_users,2276.0,179.566344,-26.0,54.0,128.0,240.0,1414.0,178.190577
sessions,2276.0,282.848858,-42.0,85.0,196.0,370.25,4030.0,300.370738
avg_session_duration,2276.0,6.846485,3.5,5.2,6.9,8.5,10.0,1.881571
month,2276.0,5.63225,1.0,2.0,5.0,9.0,12.0,3.595552
year,2276.0,2025.197715,2025.0,2025.0,2025.0,2025.0,2026.0,0.398364


In [126]:
#Outlieri sil
df_clean = df[~outlier_mask.any(axis=1)]

In [127]:
#Observing some negative values
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
date,2158,2025-08-09 14:25:28.081556992,2025-01-01 00:00:00,2025-04-19 00:00:00,2025-08-05 00:00:00,2025-11-29 00:00:00,2026-03-31 00:00:00,NaN
new_users,2158.0,55.57785,-107.0,20.0,46.0,81.0,218.0,47.361737
active_users,2158.0,152.204356,-26.0,51.25,118.5,218.0,601.0,129.089743
sessions,2158.0,236.255792,-42.0,80.0,185.5,338.0,937.0,203.629156
avg_session_duration,2158.0,6.851529,3.5,5.3,6.8,8.5,10.0,1.880802
month,2158.0,5.481001,1.0,2.0,5.0,8.0,12.0,3.486428
year,2158.0,2025.192771,2025.0,2025.0,2025.0,2025.0,2026.0,0.394566


In [128]:
#Handle with negative values
cols = ['new_users', 'active_users', 'sessions']

df_clean = df_clean[(df_clean[cols] >= 0).all(axis=1)]

In [129]:
#aggregates the total number of new users for each year and month combination.
new_users_by_year_month = df.groupby(['year', 'month'])['new_users'].sum()
print(new_users_by_year_month)

year  month
2025  1         3940.0
      2         4229.0
      3         5335.0
      4         5783.0
      5         6938.0
      6         6753.0
      7         7718.0
      8         8194.0
      9         9388.0
      10       10584.0
      11       18917.0
      12       20474.0
2026  1        12514.0
      2        11930.0
      3        14108.0
Name: new_users, dtype: float64


In [130]:
#Analysis by channel
channel_users = df.groupby('acquisition_channel')['active_users'].sum().reset_index()
channel_users = channel_users.sort_values(by='active_users', ascending=False)
#channel_users

In [131]:
#This output shows the total number of new users for each month within each year.
#it helps identify trends, seasonality, and performance changes in user acquisition over time.
new_users_by_year_month = df.groupby(['year', 'month'])['new_users'].sum()
print(new_users_by_year_month)

year  month
2025  1         3940.0
      2         4229.0
      3         5335.0
      4         5783.0
      5         6938.0
      6         6753.0
      7         7718.0
      8         8194.0
      9         9388.0
      10       10584.0
      11       18917.0
      12       20474.0
2026  1        12514.0
      2        11930.0
      3        14108.0
Name: new_users, dtype: float64


In [132]:
df.head()

,date,acquisition_channel,new_users,active_users,sessions,avg_session_duration,month,year,is_outlier
0,2025-01-01,organic,37.0,75,140.0,8.9,1,2025,False
1,2025-01-01,paid_ads,14.0,28,53.0,7.5,1,2025,False
2,2025-01-01,social_media,18.0,36,56.0,6.1,1,2025,False
3,2025-01-01,referral,30.0,70,88.0,7.5,1,2025,False
4,2025-01-01,email,12.0,33,53.0,3.8,1,2025,False


In [133]:
#Daily Active Users (DAU)
#total number of active users for each day, providing a time series view of daily user engagement.
dau = df.groupby('date')['active_users'].sum().reset_index()
dau

,date,active_users
0,2025-01-01,242
1,2025-01-02,227
2,2025-01-03,184
3,2025-01-04,307
4,2025-01-05,339
...,...,...
450,2026-03-27,1195
451,2026-03-28,1263
452,2026-03-29,1355
453,2026-03-30,1486


In [134]:
#1. DAU (Daily Active Users trend)
#the day-over-day growth rate of active users, showing how user engagement increases or decreases over time.
dau['growth_rate'] = dau['active_users'].pct_change()

In [135]:
#On which days are there sharp increases or decreases
dau

,date,active_users,growth_rate
0,2025-01-01,242,NaN
1,2025-01-02,227,-0.061983
2,2025-01-03,184,-0.189427
3,2025-01-04,307,0.668478
4,2025-01-05,339,0.104235
...,...,...,...
450,2026-03-27,1195,-0.016461
451,2026-03-28,1263,0.056904
452,2026-03-29,1355,0.072842
453,2026-03-30,1486,0.096679


In [136]:
#Let’s find the days with the fastest growth
dau.sort_values(by='growth_rate', ascending=False).head(5)

,date,active_users,growth_rate
304,2025-11-01,1852,0.919171
6,2025-01-07,432,0.674419
3,2025-01-04,307,0.668478
68,2025-03-10,587,0.586486
35,2025-02-05,455,0.531987


In [137]:
#Days to be investigated
# +20% və yuxarı artım
high_growth = dau[dau['growth_rate'] > 0.2]

# -20% və aşağı düşüş
high_drop = dau[dau['growth_rate'] < -0.2]

In [138]:
high_growth

,date,active_users,growth_rate
3,2025-01-04,307,0.668478
6,2025-01-07,432,0.674419
10,2025-01-11,451,0.281250
14,2025-01-15,407,0.267913
16,2025-01-17,394,0.304636
20,2025-01-21,386,0.349650
24,2025-01-25,375,0.530612
26,2025-01-27,378,0.251656
35,2025-02-05,455,0.531987
40,2025-02-10,430,0.207865


In [139]:
#. Sessions per user --- 1 How many sessions does a user have on average
df['sessions_per_user'] = df['sessions'] / df['active_users']
df.head()

,date,acquisition_channel,new_users,active_users,sessions,avg_session_duration,month,year,is_outlier,sessions_per_user
0,2025-01-01,organic,37.0,75,140.0,8.9,1,2025,False,1.866667
1,2025-01-01,paid_ads,14.0,28,53.0,7.5,1,2025,False,1.892857
2,2025-01-01,social_media,18.0,36,56.0,6.1,1,2025,False,1.555556
3,2025-01-01,referral,30.0,70,88.0,7.5,1,2025,False,1.257143
4,2025-01-01,email,12.0,33,53.0,3.8,1,2025,False,1.606061


In [140]:
#shows the average number of sessions per user for each acquisition channel, helping evaluate user engagement quality across channels.
channel_sessions_per_user = df.groupby('acquisition_channel')['sessions_per_user'].mean().reset_index()
print(channel_sessions_per_user)


  acquisition_channel  sessions_per_user
0               email           1.633745
1             organic           1.638165
2            paid_ads           1.562504
3            referral           1.724086
4        social_media           1.569738


In [141]:
#Total engagement
#This calculates the total engagement time by multiplying the number of sessions by the average session duration for each record.
df['total_time'] = df['sessions'] * df['avg_session_duration']

In [142]:
df.head()

,date,acquisition_channel,new_users,active_users,sessions,avg_session_duration,month,year,is_outlier,sessions_per_user,total_time
0,2025-01-01,organic,37.0,75,140.0,8.9,1,2025,False,1.866667,1246.0
1,2025-01-01,paid_ads,14.0,28,53.0,7.5,1,2025,False,1.892857,397.5
2,2025-01-01,social_media,18.0,36,56.0,6.1,1,2025,False,1.555556,341.6
3,2025-01-01,referral,30.0,70,88.0,7.5,1,2025,False,1.257143,660.0
4,2025-01-01,email,12.0,33,53.0,3.8,1,2025,False,1.606061,201.4


In [143]:
#channel's performance
channel_perf = df.groupby('acquisition_channel').agg(
    total_time=('total_time', 'sum'),
    count=('date', 'count')
).reset_index()

channel_perf = channel_perf.sort_values(by='total_time', ascending=False)

channel_perf

,acquisition_channel,total_time,count
1,organic,1486465.6,449
4,social_media,990550.8,469
2,paid_ads,947789.8,453
3,referral,513911.2,453
0,email,467797.6,452


In [144]:
#shows the average number of sessions for each day of the week, helping identify which days have higher or lower user activity.
df['weekday'] = pd.to_datetime(df['date']).dt.day_name()

df.groupby('weekday')['sessions'].mean()

,sessions
weekday,
Friday,284.603077
Monday,300.787692
Saturday,282.609231
Sunday,284.956923
Thursday,276.169231
Tuesday,278.726154
Wednesday,272.122699


In [145]:
#Anomaly days
df[df['is_outlier'] == True]

,date,acquisition_channel,new_users,active_users,sessions,avg_session_duration,month,year,is_outlier,sessions_per_user,total_time,weekday
63,2025-01-13,referral,42.0,135,2210.0,8.8,1,2025,True,16.370370,19448.0,Monday
428,2025-03-27,referral,25.0,53,950.0,7.0,3,2025,True,17.924528,6650.0,Thursday
690,2025-05-19,organic,56.0,182,2460.0,5.1,5,2025,True,13.516484,12546.0,Monday
1140,2025-08-17,organic,35.0,113,1680.0,4.0,8,2025,True,14.867257,6720.0,Sunday
1148,2025-08-18,referral,29.0,65,1090.0,7.7,8,2025,True,16.769231,8393.0,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...
2215,2026-03-20,organic,227.0,678,967.0,5.7,3,2026,True,1.426254,5511.9,Friday
2226,2026-03-22,paid_ads,237.0,736,885.0,5.2,3,2026,True,1.202446,4602.0,Sunday
2230,2026-03-23,organic,254.0,780,1313.0,7.1,3,2026,True,1.683333,9322.3,Monday
2242,2026-03-25,social_media,280.0,918,1242.0,3.9,3,2026,True,1.352941,4843.8,Wednesday


The channel that brings the most users is not necessarily the most effective — effectiveness should be measured by user engagement and time spent on the platform